# Traitement Gold pour le pôle Marketing

## Création de la session Spark et chargement des données Silver

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-gold-marketing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_reviews = spark.read.parquet("../data/silver/reviews/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_translation = spark.read.parquet("../data/silver/product_category_name_translation/")

## Création de la table Gold

In [ ]:
from pyspark.sql.functions import col

df_base = df_orders \
    .join(df_customers, "customer_id", "left") \
    .join(df_items, "order_id", "left") \
    .join(df_products, "product_id", "left") \
    .join(df_translation, "product_category_name", "left") \
    .join(df_payments, "order_id", "left") \
    .join(df_reviews, "order_id", "left") \
    .join(df_sellers, "seller_id", "left")

# Ne garder que les commandes livrées pour les analyses marketing
df_base = df_base.filter(col("order_status") == "delivered")

print("Lignes dans le DataFrame central :", df_base.count())
df_base.printSchema()

## Analyse des données pour le pôle Marketing
### Satisfaction Globale

La note moyenne de satisfaction permet d'évaluer la qualité générale du service. Une note élevée (>4) indique une bonne expérience client, tandis qu'une note basse (<3) suggère des problèmes majeurs à traiter en priorité.

**Points clés :**
- Le nombre d'avis collectés reflète le taux de participation des clients au feedback
- Cette métrique doit être suivie mensuellement pour détecter les tendances (amélioration/dégradation)
- Les baisses anormales doivent déclencher une investigation immédiate

In [ ]:
from pyspark.sql.functions import avg, count, round

df_gold_satisfaction = df_reviews \
    .join(df_orders, "order_id", "left") \
    .filter(col("order_status") == "delivered") \
    .agg(
        round(avg("review_score"), 2).alias("note_moyenne"),
        count("review_id").alias("nombre_avis"),
    )

df_gold_satisfaction.show()
df_gold_satisfaction.write.mode("overwrite").parquet("../data/gold/marketing/satisfaction_globale/")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Graphique Satisfaction Globale
satisfaction_data = df_gold_satisfaction.toPandas()

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
satisfaction_score = satisfaction_data["note_moyenne"].values[0]
nb_avis_total = int(satisfaction_data["nombre_avis"].values[0])

# Gauge chart style
colors_gauge = ["#d32f2f" if satisfaction_score < 3 else "#ff9800" if satisfaction_score < 4 else "#4caf50"]
bars = ax.barh([0], [satisfaction_score], color=colors_gauge, height=0.5, edgecolor='black', linewidth=2)

ax.set_xlim(0, 5)
ax.set_ylim(-1, 1)
ax.set_xlabel("Note Moyenne", fontsize=12, fontweight='bold')
ax.set_title("Satisfaction Globale des Clients", fontsize=14, fontweight='bold')
ax.set_yticks([])

# Ajouter le texte
ax.text(satisfaction_score/2, 0, f"{satisfaction_score}/5", 
        ha="center", va="center", fontsize=16, fontweight='bold', color='white')
ax.text(5.2, 0, f"{nb_avis_total} avis", ha="left", va="center", fontsize=10)

# Ajouter une échelle de référence
ax.axvline(x=3, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(x=4, color='orange', linestyle='--', alpha=0.5, linewidth=1)
ax.text(3, -0.7, "Problèmes", ha="center", fontsize=9, color='red')
ax.text(4, -0.7, "Moyen", ha="center", fontsize=9, color='orange')

plt.tight_layout()
plt.show()

In [ ]:
from pyspark.sql.functions import avg, count, round, when, col

df_gold_retard_satisfaction = df_base \
    .groupBy("is_late") \
    .agg(
        round(avg("review_score"), 2).alias("note_moyenne"),
        count("order_id").alias("nombre_commandes")
    ) \
    .withColumn("statut_livraison",
        when(col("is_late") == 1, "En retard").otherwise("Dans les délais")
    ) \
    .select("statut_livraison", "note_moyenne", "nombre_commandes") \
    .orderBy("note_moyenne")

df_gold_retard_satisfaction.show()
df_gold_retard_satisfaction.write.mode("overwrite").parquet("../data/gold/marketing/retard_satisfaction/")

In [ ]:
# Graphique Retard vs Satisfaction
retard_satisfaction_data = df_gold_retard_satisfaction.toPandas()

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

statuts = retard_satisfaction_data["statut_livraison"].tolist()
notes = retard_satisfaction_data["note_moyenne"].tolist()
commandes = retard_satisfaction_data["nombre_commandes"].tolist()

colors_retard = ["#4caf50", "#d32f2f"]  # Green for on-time, Red for late
bars = ax.bar(statuts, notes, color=colors_retard, edgecolor='black', linewidth=1.5, alpha=0.8)

# Ajouter les valeurs et le nombre de commandes sur les barres
for i, (bar, note, cmd) in enumerate(zip(bars, notes, commandes)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{note}/5\n({int(cmd)} cmd)', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel("Note Moyenne", fontsize=12, fontweight='bold')
ax.set_title("Impact de la Ponctualité sur la Satisfaction", fontsize=14, fontweight='bold')
ax.set_ylim(0, 5)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

### Top Catégories

Cette analyse identifie les produits moteurs du chiffre d'affaires et de la satisfaction client.

**Actions marketing recommandées :**
- **Top performers** : Augmenter la visibilité, optimiser les stocks, recruter plus de vendeurs
- **Catégories sous-performantes mais demandées** : Analyser les raisons des faibles notes (logistique, qualité produit) et corriger
- **Opportunités de cross-selling** : Promouvoir les top catégories pour attirer du trafic sur les autres produits
- **Stratégie tarifaire** : Ajuster les prix des catégories haute demande/haute satisfaction

In [ ]:
from pyspark.sql.functions import sum as _sum, count, round, col

df_gold_categories = df_base \
    .groupBy("product_category_name_english") \
    .agg(
        round(_sum("price"), 2).alias("chiffre_affaires"),
        count("order_id").alias("nombre_commandes"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .filter(col("product_category_name_english").isNotNull()) \
    .orderBy(col("chiffre_affaires").desc())

df_gold_categories.show(20)
df_gold_categories.write.mode("overwrite").parquet("../data/gold/marketing/top_categories/")

In [ ]:
# Graphique Top Catégories
categories_data = df_gold_categories.toPandas().head(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Top 10 par chiffre d'affaires
colors_ca = plt.cm.RdYlGn(categories_data['note_moyenne'] / 5)
ax1.barh(range(len(categories_data)), categories_data['chiffre_affaires'], 
         color=colors_ca, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(categories_data)))
ax1.set_yticklabels(categories_data['product_category_name_english'], fontsize=9)
ax1.set_xlabel("Chiffre d'Affaires (€)", fontsize=11, fontweight='bold')
ax1.set_title("Top 15 Catégories - Chiffre d'Affaires", fontsize=12, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# Scatter plot : CA vs Note Moyenne
scatter = ax2.scatter(categories_data['chiffre_affaires'], 
                      categories_data['note_moyenne'],
                      s=categories_data['nombre_commandes']*2,
                      alpha=0.6, c=categories_data['note_moyenne'],
                      cmap='RdYlGn', edgecolors='black', linewidth=0.5)
ax2.set_xlabel("Chiffre d'Affaires (€)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Note Moyenne", fontsize=11, fontweight='bold')
ax2.set_title("Performance: CA vs Satisfaction (taille = nb commandes)", fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3, linestyle='--')
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Note Moyenne', fontweight='bold')

plt.tight_layout()
plt.show()

### Analyse Géographique

Cette analyse révèle les disparités régionales en termes de volume et de satisfaction.

**Insights géographiques :**
- **Régions à fort potentiel** : Celles avec volume élevé ET satisfaction élevée = croissance durable
- **Régions problématiques** : Volume élevé mais satisfaction basse = risque de churn - Investigation urgente
- **Régions sous-exploitées** : Faible volume malgré bonne satisfaction = opportunité marketing
- **Logistique** : Analyser les délais de livraison par région pour expliquer les écarts de satisfaction

**Recommandations :**
- Déploiement de campagnes localisées pour les régions sous-exploitées
- Partenariats logistiques pour améliorer les services dans les régions problématiques
- Analyse concurrentielle par région

In [ ]:
from pyspark.sql.functions import count, round, sum as _sum

df_gold_geo = df_base \
    .groupBy("customer_state") \
    .agg(
        count("order_id").alias("nombre_commandes"),
        round(_sum("price"), 2).alias("chiffre_affaires"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .orderBy(col("chiffre_affaires").desc())

df_gold_geo.show(30)
df_gold_geo.write.mode("overwrite").parquet("../data/gold/marketing/geo_clients/")

In [ ]:
# Graphique Analyse Géographique
geo_data = df_gold_geo.toPandas()

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Graphique 1: Top 10 États par CA
top_geo = geo_data.head(10)
colors_geo = plt.cm.Blues(top_geo['note_moyenne'] / 5)
ax1.barh(range(len(top_geo)), top_geo['chiffre_affaires'], 
         color=colors_geo, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(top_geo)))
ax1.set_yticklabels(top_geo['customer_state'], fontsize=10)
ax1.set_xlabel("Chiffre d'Affaires (€)", fontsize=11, fontweight='bold')
ax1.set_title("Top 10 États - Chiffre d'Affaires", fontsize=12, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# Graphique 2: Nombre de commandes
ax2.bar(range(len(top_geo)), top_geo['nombre_commandes'], 
        color='#2196F3', edgecolor='black', linewidth=0.5, alpha=0.8)
ax2.set_xticks(range(len(top_geo)))
ax2.set_xticklabels(top_geo['customer_state'], rotation=45, ha='right', fontsize=10)
ax2.set_ylabel("Nombre de Commandes", fontsize=11, fontweight='bold')
ax2.set_title("Volume de Commandes par État", fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Graphique 3: Satisfaction par État
colors_sat = ['#4caf50' if x >= 4 else '#ff9800' if x >= 3 else '#d32f2f' 
              for x in top_geo['note_moyenne']]
ax3.bar(range(len(top_geo)), top_geo['note_moyenne'], 
        color=colors_sat, edgecolor='black', linewidth=0.5, alpha=0.8)
ax3.axhline(y=4, color='green', linestyle='--', alpha=0.5, label='Bon (≥4)')
ax3.axhline(y=3, color='orange', linestyle='--', alpha=0.5, label='Moyen (≥3)')
ax3.set_xticks(range(len(top_geo)))
ax3.set_xticklabels(top_geo['customer_state'], rotation=45, ha='right', fontsize=10)
ax3.set_ylabel("Note Moyenne", fontsize=11, fontweight='bold')
ax3.set_ylim(0, 5)
ax3.set_title("Satisfaction par État", fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# Graphique 4: Scatter CA vs Satisfaction
scatter = ax4.scatter(geo_data['chiffre_affaires'], 
                      geo_data['note_moyenne'],
                      s=geo_data['nombre_commandes']/2,
                      alpha=0.6, c=geo_data['note_moyenne'],
                      cmap='RdYlGn', edgecolors='black', linewidth=0.5)

# Ajouter les étiquettes des États principaux
for idx, row in geo_data.head(8).iterrows():
    ax4.annotate(row['customer_state'], 
                (row['chiffre_affaires'], row['note_moyenne']),
                fontsize=8, ha='center')

ax4.set_xlabel("Chiffre d'Affaires (€)", fontsize=11, fontweight='bold')
ax4.set_ylabel("Note Moyenne", fontsize=11, fontweight='bold')
ax4.set_title("Performance Géographique: CA vs Satisfaction", fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3, linestyle='--')
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Note Moyenne', fontweight='bold')

plt.tight_layout()
plt.show()

### Évolution Mensuelle

Cette tendance temporelle permet de comprendre la dynamique du business et les cycles saisonniers.

**Analyse des tendances :**
- **Croissance soutenue** : Stratégie efficace, continuer les investissements
- **Décroissance** : Revoir la stratégie marketing, analyser la concurrence, relancer les campagnes
- **Variations saisonnières** : Préparer les stocks et la logistique pour les pics de demande
- **Corrélation satisfaction-volume** : Si les deux baissent, problème produit/service. Si satisfaction baisse avec volume en hausse, problème de qualité due à la croissance

**Actions :**
- Planifier les campagnes promotionnelles avant les pics saisonniers
- Recruter du personnel et augmenter les capacités avant les périodes de forte demande
- Identifier les mois faibles et proposer des incitations pour les booster

In [ ]:
from pyspark.sql.functions import date_format, count, round, sum as _sum

df_gold_mensuel = df_base \
    .withColumn("mois", date_format("order_purchase_timestamp", "yyyy-MM")) \
    .groupBy("mois") \
    .agg(
        count("order_id").alias("nombre_commandes"),
        round(_sum("price"), 2).alias("chiffre_affaires"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .orderBy("mois")

df_gold_mensuel.show(24)
df_gold_mensuel.write.mode("overwrite").parquet("../data/gold/marketing/evolution_mensuelle/")

In [ ]:
# Graphique Évolution Mensuelle
mensuel_data = df_gold_mensuel.toPandas()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Graphique 1: CA et Nombre de commandes
ax1 = axes[0]
ax1_twin = ax1.twinx()

line1 = ax1.plot(range(len(mensuel_data)), mensuel_data['chiffre_affaires'], 
                 color='#1976D2', linewidth=2.5, marker='o', markersize=5, label='CA (€)')
ax1.fill_between(range(len(mensuel_data)), mensuel_data['chiffre_affaires'], 
                 alpha=0.2, color='#1976D2')

line2 = ax1_twin.plot(range(len(mensuel_data)), mensuel_data['nombre_commandes'], 
                      color='#D32F2F', linewidth=2.5, marker='s', markersize=5, label='Commandes')
ax1_twin.fill_between(range(len(mensuel_data)), mensuel_data['nombre_commandes'], 
                      alpha=0.2, color='#D32F2F')

ax1.set_ylabel("Chiffre d'Affaires (€)", fontsize=11, fontweight='bold', color='#1976D2')
ax1_twin.set_ylabel("Nombre de Commandes", fontsize=11, fontweight='bold', color='#D32F2F')
ax1.set_title("Évolution du CA et du Volume de Commandes", fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3, linestyle='--')
ax1.tick_params(axis='y', labelcolor='#1976D2')
ax1_twin.tick_params(axis='y', labelcolor='#D32F2F')

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# Graphique 2: Satisfaction mensuelle
ax2 = axes[1]
colors_trend = ['#4caf50' if x >= 4 else '#ff9800' if x >= 3 else '#d32f2f' 
                for x in mensuel_data['note_moyenne']]
ax2.bar(range(len(mensuel_data)), mensuel_data['note_moyenne'], 
        color=colors_trend, edgecolor='black', linewidth=0.5, alpha=0.8)
ax2.plot(range(len(mensuel_data)), mensuel_data['note_moyenne'], 
         color='#1976D2', linewidth=2, marker='o', markersize=5)
ax2.axhline(y=4, color='green', linestyle='--', alpha=0.5)
ax2.axhline(y=3, color='orange', linestyle='--', alpha=0.5)
ax2.set_ylabel("Note Moyenne", fontsize=11, fontweight='bold')
ax2.set_xlabel("Mois", fontsize=11, fontweight='bold')
ax2.set_ylim(0, 5)
ax2.set_title("Évolution de la Satisfaction Mensuelle", fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(mensuel_data)))
ax2.set_xticklabels(mensuel_data['mois'], rotation=45, ha='right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Fidélité Clients

Distribution du nombre de commandes par client : indicateur clé de la rétention et la LTV (Lifetime Value).

**Analyse de fidélité :**
- **Un achat** : Clients à risque de churn, taux d'acquisition inefficace - Besoin de relance urgente
- **2-3 achats** : Clients en transition, opportunité de fidélisation avec des offres loyauté
- **4+ achats** : Clients fidèles, haute LTV - À protéger, solliciter pour avis et témoignages
- **Distribution concentrée sur 1 achat** : Problème majeur de fidélisation

**Stratégies :**
- Programme de fidélité pour convertir les clients one-shot en clients réguliers
- Campagnes de réactivation 30 jours après le premier achat
- VIP program pour les clients 5+ achats (remises, livraison gratuite, early access)

In [ ]:
from pyspark.sql.functions import count, round

df_gold_fidelite = df_base \
    .filter(col("order_status") == "delivered") \
    .groupBy("customer_unique_id") \
    .agg(count("order_id").alias("nombre_commandes")) \
    .groupBy("nombre_commandes") \
    .agg(count("customer_unique_id").alias("nombre_clients")) \
    .orderBy("nombre_commandes")

df_gold_fidelite.show()
df_gold_fidelite.write.mode("overwrite").parquet("../data/gold/marketing/fidelite_clients/")

In [ ]:
# Graphique Fidélité Clients
fidelite_data = df_gold_fidelite.toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Graphique 1: Distribution du nombre de commandes par client
colors_fidelite = ['#d32f2f' if x == 1 else '#ff9800' if x <= 3 else '#4caf50' 
                   for x in fidelite_data['nombre_commandes']]
bars = ax1.bar(range(len(fidelite_data)), fidelite_data['nombre_clients'], 
               color=colors_fidelite, edgecolor='black', linewidth=0.5, alpha=0.8)

ax1.set_xlabel("Nombre de Commandes par Client", fontsize=11, fontweight='bold')
ax1.set_ylabel("Nombre de Clients", fontsize=11, fontweight='bold')
ax1.set_title("Distribution de la Fidélité Clients", fontsize=12, fontweight='bold')
ax1.set_xticks(range(len(fidelite_data)))
ax1.set_xticklabels(fidelite_data['nombre_commandes'].astype(int), fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# Ajouter valeurs sur les barres
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=9)

# Graphique 2: Pie chart des segments
one_time = fidelite_data[fidelite_data['nombre_commandes'] == 1]['nombre_clients'].values[0]
two_three = fidelite_data[fidelite_data['nombre_commandes'].isin([2, 3])]['nombre_clients'].sum()
four_plus = fidelite_data[fidelite_data['nombre_commandes'] >= 4]['nombre_clients'].sum()

segments = [one_time, two_three, four_plus]
labels = [f'Un achat\n({int(one_time):,})', 
          f'2-3 achats\n({int(two_three):,})',
          f'4+ achats\n({int(four_plus):,})']
colors_pie = ['#d32f2f', '#ff9800', '#4caf50']

wedges, texts, autotexts = ax2.pie(segments, labels=labels, colors=colors_pie,
                                     autopct='%1.1f%%', startangle=90,
                                     textprops={'fontsize': 10, 'fontweight': 'bold'})
ax2.set_title("Segmentation Clients par Fidélité", fontsize=12, fontweight='bold')

# Mettre en gras les pourcentages
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

### Acquisition Mensuelle

Nombre de nouveaux clients acquis chaque mois = indicateur de la performance des campagnes marketing.

**Analyse d'acquisition :**
- **Croissance progressive** : Stratégie marketing efficace, budgets marketing bien investis
- **Pics d'acquisition** : Identifier les campagnes/périodes qui les ont générés pour réplique
- **Creux d'acquisition** : Analyser les raisons (baisse de budget, concurrence, saisonnalité)
- **Corrélation acquisition/commandes** : Valider que les nouveaux clients restent actifs (ROI marketing)

**Métriques dérivées à calculer :**
- Coût d'acquisition par canal
- Taux de rétention 1er/2e achat des cohortes mensuelles
- LTV des clients acquis par mois/canal

**Recommandations :**
- Optimiser le budget marketing vers les canaux/périodes à meilleure conversion
- Planifier les campagnes pour lisser les creux saisonniers

In [ ]:
from pyspark.sql.functions import date_format, min as _min, count

# Première commande de chaque client = date d'acquisition
df_acquisition = df_base \
    .filter(col("order_status") == "delivered") \
    .groupBy("customer_unique_id") \
    .agg(_min("order_purchase_timestamp").alias("date_acquisition")) \
    .withColumn("mois_acquisition", date_format("date_acquisition", "yyyy-MM")) \
    .groupBy("mois_acquisition") \
    .agg(count("customer_unique_id").alias("nouveaux_clients")) \
    .orderBy("mois_acquisition")

df_acquisition.show(24)
df_acquisition.write.mode("overwrite").parquet("../data/gold/marketing/acquisition_mensuelle/")

In [ ]:
# Graphique Acquisition Mensuelle
acquisition_data = df_acquisition.toPandas()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Graphique 1: Nombre de nouveaux clients par mois
colors_acq = plt.cm.Blues(acquisition_data['nouveaux_clients'] / acquisition_data['nouveaux_clients'].max())
bars = ax1.bar(range(len(acquisition_data)), acquisition_data['nouveaux_clients'],
               color=colors_acq, edgecolor='black', linewidth=0.5, alpha=0.8)
ax1.plot(range(len(acquisition_data)), acquisition_data['nouveaux_clients'],
         color='#1976D2', linewidth=2, marker='o', markersize=5)
ax1.fill_between(range(len(acquisition_data)), acquisition_data['nouveaux_clients'],
                 alpha=0.2, color='#1976D2')
ax1.set_ylabel("Nouveaux Clients", fontsize=11, fontweight='bold')
ax1.set_title("Évolution de l'Acquisition Mensuelle", fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Ajouter la moyenne mobile
if len(acquisition_data) >= 3:
    moving_avg = acquisition_data['nouveaux_clients'].rolling(window=3, center=True).mean()
    ax1.plot(range(len(moving_avg)), moving_avg, color='red', linewidth=2, 
             linestyle='--', label='Moyenne mobile (3 mois)', alpha=0.7)
    ax1.legend()

# Graphique 2: Variation mensuelle (%)
ax2_data = acquisition_data.copy()
ax2_data['variation'] = ax2_data['nouveaux_clients'].pct_change() * 100

colors_var = ['#4caf50' if x >= 0 else '#d32f2f' if x != x else '#999999' 
              for x in ax2_data['variation']]
ax2.bar(range(len(ax2_data)), ax2_data['variation'], 
        color=colors_var, edgecolor='black', linewidth=0.5, alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.set_ylabel("Variation (%)", fontsize=11, fontweight='bold')
ax2.set_xlabel("Mois", fontsize=11, fontweight='bold')
ax2.set_title("Variation Mensuelle de l'Acquisition", fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(acquisition_data)))
ax2.set_xticklabels(acquisition_data['mois_acquisition'], rotation=45, ha='right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### NPS par Catégorie

Le Net Promoter Score mesure la propension des clients à recommander un produit (proxy). Indicateur clé de satisfaction et de fidélité.

**Lecture du NPS :**
- **NPS > 50** : Excellent, clients très satisfaits, fort potentiel de recommandation
- **NPS 0-50** : Bon, mais opportunités d'amélioration
- **NPS < 0** : Critique, risque de détracteurs qui parlent mal du produit
- **Nombre d'avis < 50** : Données insuffisamment significatives statistiquement

**Catégories enchanteurs (NPS haut) :**
- Augmenter le marketing et la visibilité
- Cas d'études à partager avec le reste de l'org
- Modèle à répliquer pour les autres catégories

**Catégories à problème (NPS bas) :**
- Investigation immédiate : qualité produit ? logistique ? prix ? 
- Focus sur l'expérience client avant toute promotion
- Considérer l'exclusion si pas d'amélioration rapide

In [ ]:
from pyspark.sql.functions import avg, count, round, col, when, sum as _sum

df_nps = df_base \
    .filter(col("review_score").isNotNull()) \
    .withColumn("promoteur",   when(col("review_score") == 5, 1).otherwise(0)) \
    .withColumn("detracteur",  when(col("review_score") <= 2, 1).otherwise(0)) \
    .groupBy("product_category_name_english") \
    .agg(
        count("review_id").alias("nb_avis"),
        round(avg("review_score"), 2).alias("note_moyenne"),
        round(
            (_sum("promoteur") - _sum("detracteur")) / count("review_id") * 100, 1
        ).alias("nps_score")  # proxy NPS : % promoteurs - % détracteurs
    ) \
    .filter(col("product_category_name_english").isNotNull()) \
    .filter(col("nb_avis") >= 50)  # seuil de significativité statistique

# Top catégories enchanteurs vs décevantes
print("=== Top catégories satisfaisantes ===")
df_nps.orderBy(col("nps_score").desc()).show(10)

print("=== Catégories à problème ===")
df_nps.orderBy(col("nps_score").asc()).show(10)

df_nps.write.mode("overwrite").parquet("../data/gold/marketing/nps_par_categorie/")

In [ ]:
# Graphique NPS par Catégorie
nps_data = df_nps.toPandas().sort_values('nps_score', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Graphique 1: NPS Score - Barres horizontales
colors_nps = ['#d32f2f' if x < 0 else '#ff9800' if x < 50 else '#4caf50' 
              for x in nps_data['nps_score']]
bars = ax1.barh(range(len(nps_data)), nps_data['nps_score'],
                color=colors_nps, edgecolor='black', linewidth=0.5, alpha=0.8)
ax1.set_yticks(range(len(nps_data)))
ax1.set_yticklabels(nps_data['product_category_name_english'], fontsize=9)
ax1.set_xlabel("NPS Score", fontsize=11, fontweight='bold')
ax1.set_title("NPS par Catégorie (Promoteurs % - Détracteurs %)", fontsize=12, fontweight='bold')
ax1.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax1.axvline(x=50, color='green', linestyle='--', alpha=0.5)
ax1.grid(axis='x', alpha=0.3)

# Ajouter les valeurs
for i, (bar, val) in enumerate(zip(bars, nps_data['nps_score'])):
    ax1.text(val + 2 if val >= 0 else val - 2, i, f'{val:.1f}', 
            ha='left' if val >= 0 else 'right', va='center', fontsize=8, fontweight='bold')

# Graphique 2: Scatter - NPS vs Volume
scatter = ax2.scatter(nps_data['nb_avis'], nps_data['nps_score'],
                     s=100, c=nps_data['note_moyenne'],
                     cmap='RdYlGn', vmin=1, vmax=5,
                     alpha=0.6, edgecolors='black', linewidth=0.5)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='NPS critique')
ax2.axhline(y=50, color='green', linestyle='--', alpha=0.5, label='NPS excellent')
ax2.set_xlabel("Nombre d'Avis", fontsize=11, fontweight='bold')
ax2.set_ylabel("NPS Score", fontsize=11, fontweight='bold')
ax2.set_title("NPS vs Volume d'Avis (couleur = note moyenne)", fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3, linestyle='--')
ax2.legend()
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Note Moyenne', fontweight='bold')

plt.tight_layout()
plt.show()

### Saisonnalité des Catégories

Identifier les patterns saisonniers par catégorie permet d'optimiser les stocks, budgets et planification.

**Patterns typiques :**
- **Électronique** : pics à Noël/Black Friday, creux en janvier
- **Mode** : pics saisonniers (été, hiver), soldes anticipées
- **Cadeaux/Jouets** : pics à Noël et anniversaires (concentrés)
- **Aliments/Basiques** : demande relativement stable

**Opportunités marketing :**
- **Anticiper les pics** : Augmenter les stocks, budgets marketing et partenaires 1-2 mois avant
- **Booster les creux** : Promotions, bundle deals, cross-selling avec catégories en pic
- **Planification des campagnes** : Aligner avec les comportements d'achat naturels
- **Recrutement logistique** : Préparer les effectifs pour les périodes de forte activité

In [ ]:
from pyspark.sql.functions import month, count, col

df_saisonnalite = df_base \
    .withColumn("mois", month("order_purchase_timestamp")) \
    .groupBy("product_category_name_english", "mois") \
    .agg(count("order_id").alias("nb_commandes")) \
    .filter(col("product_category_name_english").isNotNull()) \
    .orderBy("product_category_name_english", "mois")

# Focus sur les top 5 catégories
top5_categories = [
    row["product_category_name_english"]
    for row in df_base
        .groupBy("product_category_name_english")
        .count()
        .orderBy(col("count").desc())
        .limit(5)
        .collect()
]

df_saisonnalite \
    .filter(col("product_category_name_english").isin(top5_categories)) \
    .show(60)

df_saisonnalite.write.mode("overwrite").parquet("../data/gold/marketing/saisonnalite_categories/")

In [ ]:
# Graphique Saisonnalité des Catégories
saisonnalite_data = df_saisonnalite.toPandas()
top5_categories = [
    row["product_category_name_english"]
    for row in df_base
        .groupBy("product_category_name_english")
        .count()
        .orderBy(col("count").desc())
        .limit(5)
        .collect()
]

saisonnalite_top5 = saisonnalite_data[
    saisonnalite_data['product_category_name_english'].isin(top5_categories)
].copy()

fig, ax = plt.subplots(figsize=(14, 7))

# Heatmap-like plot avec des courbes
month_names = {1: 'Jan', 2: 'Fév', 3: 'Mar', 4: 'Avr', 5: 'Mai', 6: 'Juin',
               7: 'Juil', 8: 'Aoû', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Déc'}

colors_cat = plt.cm.Set2(range(len(top5_categories)))

for idx, cat in enumerate(top5_categories):
    cat_data = saisonnalite_top5[saisonnalite_top5['product_category_name_english'] == cat].sort_values('mois')
    ax.plot(cat_data['mois'], cat_data['nb_commandes'], 
           marker='o', linewidth=2.5, markersize=7, label=cat, color=colors_cat[idx], alpha=0.8)

ax.set_xlabel("Mois", fontsize=11, fontweight='bold')
ax.set_ylabel("Nombre de Commandes", fontsize=11, fontweight='bold')
ax.set_title("Saisonnalité des Top 5 Catégories", fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels([month_names[i] for i in range(1, 13)], fontsize=10)
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

### Pénétration Géographique par Catégorie

Comprendre quelles catégories dominent dans chaque région permet des stratégies marketing ultra-ciblées.

**Insights géo-catégories :**
- **Part de marché élevée** : Catégorie leader régionale, forte demande locale
- **Part de marché faible** : Potentiel d'expansion ou déclin - à investiguer
- **Disparités régionales** : Certaines catégories sont régionales (ex: vêtements côtiers) vs nationales

**Stratégies par région :**
- **Régions dominées par une catégorie** : Partenariats avec leaders régionaux, sponsoring événementiel local
- **Régions diversifiées** : Approche généraliste, cross-selling opportuniste
- **Catégories sous-pénétrées** : Campagnes ciblées pour tester la demande locale

**Recommandations :**
- Créer des catalogues personnalisés par région (produits en vedette régionalisée)
- Adapter les campagnes emails/SMS par région et catégorie dominante
- Analyser pourquoi certaines catégories ne percent pas dans certaines régions

In [ ]:
from pyspark.sql.functions import count, col, round, sum as _sum

# Commandes totales par état
df_total_par_etat = df_base \
    .groupBy("customer_state") \
    .agg(count("order_id").alias("total_commandes_etat"))

# Commandes par état ET par catégorie
df_geo_categorie = df_base \
    .filter(col("product_category_name_english").isNotNull()) \
    .groupBy("customer_state", "product_category_name_english") \
    .agg(count("order_id").alias("nb_commandes")) \
    .join(df_total_par_etat, "customer_state") \
    .withColumn(
        "part_de_marche_pct",
        round(col("nb_commandes") / col("total_commandes_etat") * 100, 2)
    ) \
    .orderBy("customer_state", col("nb_commandes").desc())

df_geo_categorie.show(20)
df_geo_categorie.write.mode("overwrite").parquet("../data/gold/marketing/penetration_geo_categorie/")

In [ ]:
# Graphique Pénétration Géographique par Catégorie
geo_categorie_data = df_geo_categorie.toPandas()

# Focus sur les états principaux et leurs top catégories
top_states = geo_data['customer_state'].head(8).tolist()
geo_cat_top = geo_categorie_data[geo_categorie_data['customer_state'].isin(top_states)].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, state in enumerate(top_states):
    ax = axes[idx // 2] if idx < 4 else axes[idx - 4]
    
    state_data = geo_cat_top[geo_cat_top['customer_state'] == state].head(10)
    colors_part = plt.cm.Set3(range(len(state_data)))
    
    bars = ax.barh(range(len(state_data)), state_data['part_de_marche_pct'],
                  color=colors_part, edgecolor='black', linewidth=0.5, alpha=0.8)
    ax.set_yticks(range(len(state_data)))
    ax.set_yticklabels(state_data['product_category_name_english'], fontsize=8)
    ax.set_xlabel("Part de Marché (%)", fontsize=10, fontweight='bold')
    ax.set_title(f"État: {state}", fontsize=11, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Ajouter les valeurs
    for bar, val in zip(bars, state_data['part_de_marche_pct']):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
               ha='left', va='center', fontsize=8)

plt.suptitle("Pénétration des Catégories par État (Top 8 États)", 
            fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### Affinité Produit (Market Basket Analysis)

Identifier les paires de catégories achetées ensemble permet d'optimiser le revenue par commande.

**Paires à haut potentiel :**
- **Fréquence élevée** : Combinaisons naturelles (ex: lit + draps, caméra + lentilles)
- **Opportunités de cross-sell** : Les clients achètent X, ils devraient acheter Y
- **Bundle opportuns** : Créer des packs de catégories complémentaires pour boost AOV

**Stratégies de cross-sell :**
1. **Sur le site** : Recommandations produits lors de la consultation/panier
2. **Campagnes email** : "Les clients qui ont acheté X achètent aussi Y"
3. **Packaging** : Insérer des coupons de réduction pour produits complémentaires
4. **Offres combo** : Réduction si achat de 2 catégories ensemble

**Gains potentiels :**
- Augmentation du panier moyen (AOV) sans hausse du trafic
- Réduction de la friction entre catégories concurrentes
- Amélioration de la satisfaction (clients trouvent tout en un lieu)

In [ ]:
from pyspark.sql.functions import col, count

# Catégories par commande
df_cat_par_commande = df_items \
    .join(df_products, "product_id") \
    .join(df_translation, "product_category_name") \
    .select("order_id", "product_category_name_english") \
    .distinct()

# Auto-jointure pour trouver les paires
df_affinite = df_cat_par_commande.alias("a") \
    .join(
        df_cat_par_commande.alias("b"),
        on=(col("a.order_id") == col("b.order_id")) &
           (col("a.product_category_name_english") < col("b.product_category_name_english"))
    ) \
    .groupBy(
        col("a.product_category_name_english").alias("categorie_1"),
        col("b.product_category_name_english").alias("categorie_2")
    ) \
    .agg(count("a.order_id").alias("achetes_ensemble")) \
    .orderBy(col("achetes_ensemble").desc())

print("=== Top paires de catégories achetées ensemble ===")
df_affinite.show(20)
df_affinite.write.mode("overwrite").parquet("../data/gold/marketing/affinite_produit/")

## Tableau de Synthèse - Indicateurs Clés Marketing

In [ ]:
# Synthèse finale - Tableau KPI
import pandas as pd
import matplotlib.pyplot as plt

# Récupérer les données principales
satisfaction_score = df_gold_satisfaction.toPandas()["note_moyenne"].values[0]
nb_avis = int(df_gold_satisfaction.toPandas()["nombre_avis"].values[0])

retard_data = df_gold_retard_satisfaction.toPandas()
satisfaction_on_time = retard_data[retard_data["statut_livraison"] == "Dans les délais"]["note_moyenne"].values[0]
satisfaction_late = retard_data[retard_data["statut_livraison"] == "En retard"]["note_moyenne"].values[0]

categories_data = df_gold_categories.toPandas()
total_revenue = categories_data['chiffre_affaires'].sum()
total_orders = int(categories_data['nombre_commandes'].sum())
top_category = categories_data.iloc[0]['product_category_name_english']
top_category_revenue = categories_data.iloc[0]['chiffre_affaires']

geo_data_pd = df_gold_geo.toPandas()
total_customers = len(df_customers.distinct().collect())
top_state = geo_data_pd.iloc[0]['customer_state']
top_state_revenue = geo_data_pd.iloc[0]['chiffre_affaires']

fidelite_data_pd = df_gold_fidelite.toPandas()
one_time_pct = (fidelite_data_pd[fidelite_data_pd['nombre_commandes'] == 1]['nombre_clients'].sum() / 
                fidelite_data_pd['nombre_clients'].sum() * 100)
repeat_customers = fidelite_data_pd[fidelite_data_pd['nombre_commandes'] > 1]['nombre_clients'].sum()
repeat_rate = (repeat_customers / fidelite_data_pd['nombre_clients'].sum()) * 100

acquisition_data_pd = df_acquisition.toPandas()
new_customers_last_month = int(acquisition_data_pd.iloc[-1]['nouveaux_clients'])
avg_monthly_acquisition = int(acquisition_data_pd['nouveaux_clients'].mean())

nps_data_pd = df_nps.toPandas()
avg_nps = nps_data_pd['nps_score'].mean()
high_nps_count = len(nps_data_pd[nps_data_pd['nps_score'] > 50])
low_nps_count = len(nps_data_pd[nps_data_pd['nps_score'] < 0])

mensuel_data_pd = df_gold_mensuel.toPandas()
revenue_last_month = mensuel_data_pd.iloc[-1]['chiffre_affaires']
satisfaction_last_month = mensuel_data_pd.iloc[-1]['note_moyenne']
orders_last_month = int(mensuel_data_pd.iloc[-1]['nombre_commandes'])

affinite_data_pd = df_affinite.toPandas()
top_affinity = affinite_data_pd.iloc[0]
avg_order_value = total_revenue / total_orders

# Créer le tableau KPI
kpi_data = {
    'Catégorie': [
        '📊 SATISFACTION',
        '📊 SATISFACTION',
        '📊 SATISFACTION',
        '📊 SATISFACTION',
        '💰 CHIFFRE D\'AFFAIRES',
        '💰 CHIFFRE D\'AFFAIRES',
        '💰 CHIFFRE D\'AFFAIRES',
        '💰 CHIFFRE D\'AFFAIRES',
        '🌍 GÉOGRAPHIE',
        '🌍 GÉOGRAPHIE',
        '👥 CLIENTS',
        '👥 CLIENTS',
        '👥 CLIENTS',
        '👥 CLIENTS',
        '📈 ACQUISITION',
        '📈 ACQUISITION',
        '⭐ NPS',
        '⭐ NPS',
        '⭐ NPS',
        '🔗 AFFINITÉS',
    ],
    'Indicateur': [
        'Note Moyenne',
        'Avis Collectés',
        'Satisfaction (Livraison à Temps)',
        'Satisfaction (Retard)',
        'CA Total',
        'Nombre de Commandes',
        'Catégorie Top (CA)',
        'Panier Moyen',
        'État Principal',
        'CA État Top',
        'Nombre de Clients',
        'Clients One-Time',
        'Taux de Récurrence',
        'Clients Récurrents',
        'Nouveaux Clients (Mois)',
        'Acquisition Mensuelle Moyenne',
        'NPS Moyen',
        'Catégories Excellentes (NPS>50)',
        'Catégories Critiques (NPS<0)',
        'Top Affinity'
    ],
    'Valeur': [
        f"{satisfaction_score}/5",
        f"{nb_avis:,}",
        f"{satisfaction_on_time}/5",
        f"{satisfaction_late}/5",
        f"€{total_revenue:,.0f}",
        f"{total_orders:,}",
        f"{top_category}",
        f"€{avg_order_value:.2f}",
        f"{top_state}",
        f"€{top_state_revenue:,.0f}",
        f"{total_customers:,}",
        f"{int(fidelite_data_pd[fidelite_data_pd['nombre_commandes'] == 1]['nombre_clients'].sum()):,} ({one_time_pct:.1f}%)",
        f"{repeat_rate:.1f}%",
        f"{int(repeat_customers):,}",
        f"{new_customers_last_month:,}",
        f"{avg_monthly_acquisition:,}",
        f"{avg_nps:.1f}",
        f"{high_nps_count}",
        f"{low_nps_count}",
        f"{top_affinity['categorie_1']} + {top_affinity['categorie_2']} ({int(top_affinity['achetes_ensemble'])} fois)"
    ]
}

kpi_df = pd.DataFrame(kpi_data)

# Afficher le tableau avec formatage
fig, ax = plt.subplots(figsize=(16, 12))
ax.axis('tight')
ax.axis('off')

# Créer le tableau
table = ax.table(cellText=kpi_df.values, colLabels=kpi_df.columns,
                cellLoc='left', loc='center',
                colWidths=[0.2, 0.35, 0.45])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Formatage du header
for i in range(len(kpi_df.columns)):
    table[(0, i)].set_facecolor('#1976D2')
    table[(0, i)].set_text_props(weight='bold', color='white', fontsize=11)

# Formatage alternant les couleurs de lignes par catégorie
colors_map = {
    '📊 SATISFACTION': '#E3F2FD',
    '💰 CHIFFRE D\'AFFAIRES': '#F3E5F5',
    '🌍 GÉOGRAPHIE': '#E8F5E9',
    '👥 CLIENTS': '#FFF3E0',
    '📈 ACQUISITION': '#FCE4EC',
    '⭐ NPS': '#F0F4C3',
    '🔗 AFFINITÉS': '#E0F2F1'
}

for i in range(1, len(kpi_df) + 1):
    category = kpi_df.iloc[i-1]['Catégorie']
    color = colors_map.get(category, '#FFFFFF')
    
    for j in range(len(kpi_df.columns)):
        table[(i, j)].set_facecolor(color)
        table[(i, j)].set_text_props(fontsize=10)
        
        # Mettre en gras la première colonne (Catégorie)
        if j == 0:
            table[(i, j)].set_text_props(weight='bold', fontsize=10)

plt.title("Synthèse des Indicateurs Clés Marketing", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("RÉSUMÉ EXÉCUTIF - MARKETING")
print("="*80)
print(f"\n✓ Satisfaction Globale: {satisfaction_score}/5 ({nb_avis:,} avis)")
print(f"✓ Impact de la Ponctualité: +{satisfaction_on_time - satisfaction_late:.2f} points d'écart")
print(f"✓ Chiffre d'Affaires: €{total_revenue:,.0f} ({total_orders:,} commandes)")
print(f"✓ Panier Moyen: €{avg_order_value:.2f}")
print(f"✓ Taux de Récurrence: {repeat_rate:.1f}% ({int(repeat_customers):,} clients fidèles)")
print(f"✓ Acquisition: {avg_monthly_acquisition:,} nouveaux clients/mois")
print(f"✓ NPS Moyen: {avg_nps:.1f} (Excellentes: {high_nps_count}, Critiques: {low_nps_count})")
print("="*80)

In [ ]:
# Graphique Affinité Produit
affinite_data = df_affinite.toPandas().head(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Graphique 1: Top paires - barres horizontales
pair_labels = [f"{row['categorie_1'][:20]}...\n+ {row['categorie_2'][:20]}..." 
               if len(row['categorie_1']) > 20 else f"{row['categorie_1']}\n+ {row['categorie_2']}"
               for _, row in affinite_data.iterrows()]

colors_affinity = plt.cm.Spectral(affinite_data['achetes_ensemble'] / affinite_data['achetes_ensemble'].max())
bars = ax1.barh(range(len(affinite_data)), affinite_data['achetes_ensemble'],
               color=colors_affinity, edgecolor='black', linewidth=0.5, alpha=0.8)
ax1.set_yticks(range(len(affinite_data)))
ax1.set_yticklabels([f"{row['categorie_1']}\n+ {row['categorie_2']}" 
                     for _, row in affinite_data.iterrows()], fontsize=8)
ax1.set_xlabel("Nombre de Commandes Conjointes", fontsize=11, fontweight='bold')
ax1.set_title("Top 15 Paires de Catégories Achetées Ensemble", fontsize=12, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# Ajouter les valeurs
for bar, val in zip(bars, affinite_data['achetes_ensemble']):
    ax1.text(val + 20, bar.get_y() + bar.get_height()/2, f'{int(val):,}',
            ha='left', va='center', fontsize=8, fontweight='bold')

# Graphique 2: Distribution des forces d'affinité
all_affinite = df_affinite.toPandas()
ax2.hist(all_affinite['achetes_ensemble'], bins=30, color='#2196F3', 
        edgecolor='black', alpha=0.7)
ax2.axvline(x=affinite_data['achetes_ensemble'].mean(), color='red', 
           linestyle='--', linewidth=2, label=f"Moyenne: {affinite_data['achetes_ensemble'].mean():.0f}")
ax2.set_xlabel("Nombre de Commandes Conjointes", fontsize=11, fontweight='bold')
ax2.set_ylabel("Nombre de Paires", fontsize=11, fontweight='bold')
ax2.set_title("Distribution des Affinités de Produits", fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()